
# Thesis-Ready Multimodal Sentiment Analysis
## Uncertainty-Aware Co-Attention + Evidential Deep Learning (EDL)

Dataset:
- `D:/MVSA_SINGLE`


In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================

class CFG:

    # =========================
    # PATH
    # =========================
    ROOT_DIR = r"D:/MVSA_SINGLE"
    DATA_DIR = r"D:/MVSA_SINGLE/data"
    LABEL_PATH = r"D:/MVSA_SINGLE/labelResultAllFinal.txt"


    # =========================
    # DEVICE
    # =========================
    DEVICE = "cuda"


In [14]:
import pandas as pd
import os
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(CFG.LABEL_PATH, header=0, sep=',')
df.columns = ["id", "text_label", "image_label", "final_label"]

def is_valid(row):

    if row["text_label"] == "positive" and row["image_label"] == "negative":
        return False

    if row["text_label"] == "negative" and row["image_label"] == "positive":
        return False

    return True

df = df[df.apply(is_valid, axis=1)]
df = df.reset_index(drop=True)

print(f"Dataset size after filtering: {len(df)}")

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

df["label"] = df["final_label"].map(label_map)

# Load text file dengan better error handling
def load_text(sample_id):
    path = os.path.join(CFG.DATA_DIR, f"{sample_id}.txt")

    encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]

    for encoding in encodings:
        try:
            with open(path, "r", encoding=encoding) as f:
                text = f.read().strip()
                if text:  # Jika text berhasil dibaca dan tidak kosong
                    return text
        except FileNotFoundError:
            continue
        except Exception as e:
            continue

    # Jika semua encoding gagal atau file tidak ada
    return ""

# Track failed samples for debugging
failed_samples = []

df["text"] = df["id"].apply(load_text)

# Hitung empty text
empty_text_count = (df["text"] == "").sum()
print(f"\n{'='*60}")
print(f"PREPROCESSING STATISTICS:")
print(f"{'='*60}")
print(f"Total samples: {len(df)}")
print(f"Samples with EMPTY text: {empty_text_count}")
print(f"Samples with VALID text: {len(df) - empty_text_count}")
print(f"Percentage of empty text: {(empty_text_count/len(df)*100):.2f}%")
print(f"{'='*60}\n")

if empty_text_count > 0:
    print("IDs with empty text:")
    empty_ids = df[df["text"] == ""]["id"].tolist()
    for idx in empty_ids[:10]:  # Show first 10
        print(f"  - {idx}")
    if len(empty_ids) > 10:
        print(f"  ... and {len(empty_ids) - 10} more")
    print()

# image path
df["image_path"] = df["id"].apply(
    lambda x: os.path.join(CFG.DATA_DIR, f"{x}.jpg")
)

df.head()


Dataset size after filtering: 4511

PREPROCESSING STATISTICS:
Total samples: 4511
Samples with EMPTY text: 0
Samples with VALID text: 4511
Percentage of empty text: 0.00%



,id,text_label,image_label,final_label,label,text,image_path
0,1,neutral,positive,positive,2,How I feel today #legday #jelly #aching #gym,D:/MVSA_SINGLE/data\1.jpg
1,2,neutral,positive,positive,2,grattis min griskulting!!!???? va bara tvungen...,D:/MVSA_SINGLE/data\2.jpg
2,3,neutral,positive,positive,2,RT @polynminion: The moment I found my favouri...,D:/MVSA_SINGLE/data\3.jpg
3,4,positive,positive,positive,2,#escort We have a young and energetic team and...,D:/MVSA_SINGLE/data\4.jpg
4,5,positive,positive,positive,2,"RT @chrisashaffer: Went to SSC today to be a ""...",D:/MVSA_SINGLE/data\5.jpg


## Fase 1 - Environment, Hyperparameters, dan Data Pipeline

Cell berikut melanjutkan template loader dataset di atas. Dataset tetap bersumber dari `df` yang sudah memiliki kolom `text`, `image_path`, dan `label`.


In [ ]:
# ============================================================
# CELL 3: IMPORTS, SEED, DEVICE, AND HYPERPARAMETERS
# ============================================================
import copy
import math
import os
import random
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class HP:
    # Data
    RANDOM_STATE: int = 42
    TEST_SIZE: float = 0.10
    VAL_SIZE: float = 0.10
    NUM_CLASSES: int = 3
    MAX_TEXT_LEN: int = 128
    IMG_SIZE: int = 224
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True

    # Backbones
    TEXT_MODEL_NAME: str = "roberta-base"
    PROJ_DIM: int = 256
    ATTN_HEADS: int = 8
    DROPOUT: float = 0.30
    USE_PRETRAINED_IMAGE: bool = True

    # Optimization
    BATCH_SIZE: int = 16
    MAX_EPOCHS: int = 20
    BACKBONE_LR: float = 2e-5
    HEAD_LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-2
    GRAD_CLIP_NORM: float = 1.0
    EARLY_STOP_PATIENCE: int = 5
    BACKBONE_FREEZE_EPOCHS: int = 0

    # EDL objective
    AUX_LOSS_WEIGHT: float = 0.30
    KL_WEIGHT: float = 1.0
    KL_ANNEALING_EPOCHS: int = 10
    ORTHO_LOSS_WEIGHT: float = 1e-3

    # Optional artifacts
    BEST_MODEL_PATH: str = "best_decoupled_bca_edl.pt"
    HISTORY_PATH: str = "training_history_decoupled_bca_edl.npy"


set_seed(HP.RANDOM_STATE)
device = torch.device(CFG.DEVICE if str(CFG.DEVICE).lower() == "cuda" and torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Text backbone: {HP.TEXT_MODEL_NAME} | Image backbone: ResNet50")


In [ ]:
# ============================================================
# CELL 4: MVSA DATASET AND DATALOADERS
# ============================================================

required_columns = {"text", "image_path", "label"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset template is missing required columns: {sorted(missing_columns)}")

work_df = df.dropna(subset=["label"]).copy()
work_df["label"] = work_df["label"].astype(int)
work_df["text"] = work_df["text"].fillna("").astype(str)

train_df, test_df = train_test_split(
    work_df,
    test_size=HP.TEST_SIZE,
    random_state=HP.RANDOM_STATE,
    stratify=work_df["label"],
)
train_df, val_df = train_test_split(
    train_df,
    test_size=HP.VAL_SIZE / (1.0 - HP.TEST_SIZE),
    random_state=HP.RANDOM_STATE,
    stratify=train_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train label distribution:")
print(train_df["label"].value_counts().sort_index())

tokenizer = AutoTokenizer.from_pretrained(HP.TEXT_MODEL_NAME)

train_transform = transforms.Compose([
    transforms.Resize((HP.IMG_SIZE + 32, HP.IMG_SIZE + 32)),
    transforms.RandomResizedCrop(HP.IMG_SIZE, scale=(0.80, 1.00)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((HP.IMG_SIZE, HP.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class MVSADataset(Dataset):
    def __init__(self, dataframe, tokenizer, image_transform, max_text_len=HP.MAX_TEXT_LEN):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"] if isinstance(row["text"], str) else ""

        encoded = self.tokenizer(
            text,
            max_length=self.max_text_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        image_path = row["image_path"]
        try:
            image = Image.open(image_path).convert("RGB")
            image = self.image_transform(image)
        except Exception:
            image = torch.zeros(3, HP.IMG_SIZE, HP.IMG_SIZE, dtype=torch.float32)

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "image": image,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "sample_id": row["id"] if "id" in self.df.columns else idx,
        }


train_dataset = MVSADataset(train_df, tokenizer, train_transform)
val_dataset = MVSADataset(val_df, tokenizer, eval_transform)
test_dataset = MVSADataset(test_df, tokenizer, eval_transform)

class_counts = train_df["label"].value_counts().sort_index().reindex(range(HP.NUM_CLASSES), fill_value=1).values.astype(np.float32)
class_weights_np = (1.0 / np.sqrt(class_counts))
class_weights_np = class_weights_np / class_weights_np.sum() * HP.NUM_CLASSES
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

train_loader = DataLoader(
    train_dataset,
    batch_size=HP.BATCH_SIZE,
    shuffle=True,
    num_workers=HP.NUM_WORKERS,
    pin_memory=HP.PIN_MEMORY and device.type == "cuda",
)
val_loader = DataLoader(
    val_dataset,
    batch_size=HP.BATCH_SIZE,
    shuffle=False,
    num_workers=HP.NUM_WORKERS,
    pin_memory=HP.PIN_MEMORY and device.type == "cuda",
)
test_loader = DataLoader(
    test_dataset,
    batch_size=HP.BATCH_SIZE,
    shuffle=False,
    num_workers=HP.NUM_WORKERS,
    pin_memory=HP.PIN_MEMORY and device.type == "cuda",
)

print(f"Class weights: {class_weights.detach().cpu().numpy().round(4).tolist()}")
print(f"DataLoaders ready | train batches: {len(train_loader)}")


## Fase 2-6 - Decoupled Bi-directional Cross-Attention + EDL Fusion

Model ini mengekstraksi token teks RoBERTa dan token spasial ResNet50, memisahkan fitur sentimen/background, menjalankan cross-attention dua arah, lalu menggabungkan dua opini evidential menggunakan Dempster-Shafer.


In [ ]:
# ============================================================
# CELL 5: MODEL ARCHITECTURE - Decoupled BCA + EDL
# ============================================================

def masked_mean_pool(x, mask=None, eps=1e-8):
    if mask is None:
        return x.mean(dim=1)
    mask = mask.unsqueeze(-1).to(dtype=x.dtype)
    return (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=eps)


def orthogonal_penalty(sentiment_features, background_features, mask=None, eps=1e-8):
    sent = F.normalize(sentiment_features, dim=-1, eps=eps)
    back = F.normalize(background_features, dim=-1, eps=eps)
    penalty = (sent * back).sum(dim=-1).pow(2)
    if mask is not None:
        mask = mask.to(dtype=penalty.dtype)
        return (penalty * mask).sum() / mask.sum().clamp(min=eps)
    return penalty.mean()


class FeatureDecoupler(nn.Module):
    # Split modality features into sentiment and background subspaces.
    def __init__(self, dim, dropout=0.30):
        super().__init__()
        self.sentiment_mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
        )
        self.background_mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
        )

    def forward(self, x):
        return self.sentiment_mlp(x), self.background_mlp(x)


class DirectionalEDLHead(nn.Module):
    # Feature vector -> non-negative evidence -> Dirichlet opinion.
    def __init__(self, dim, num_classes, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_classes),
        )
        self.softplus = nn.Softplus()
        self.num_classes = num_classes

    def forward(self, x):
        evidence = self.softplus(self.net(x))
        alpha = evidence + 1.0
        S = alpha.sum(dim=-1, keepdim=True)
        belief = evidence / S
        uncertainty = self.num_classes / S
        return alpha, belief, uncertainty, evidence


class DempsterShaferFusion(nn.Module):
    # Fuse two evidential opinions with Dempster-Shafer combination.
    def __init__(self, num_classes, eps=1e-8):
        super().__init__()
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, alpha1, alpha2):
        evidence1 = alpha1 - 1.0
        evidence2 = alpha2 - 1.0
        S1 = alpha1.sum(dim=-1, keepdim=True)
        S2 = alpha2.sum(dim=-1, keepdim=True)

        belief1 = evidence1 / S1.clamp(min=self.eps)
        belief2 = evidence2 / S2.clamp(min=self.eps)
        uncertainty1 = self.num_classes / S1.clamp(min=self.eps)
        uncertainty2 = self.num_classes / S2.clamp(min=self.eps)

        pairwise_belief = torch.bmm(belief1.unsqueeze(2), belief2.unsqueeze(1))
        total_mass = pairwise_belief.sum(dim=(1, 2), keepdim=True).squeeze(-1)
        agreement_mass = torch.diagonal(pairwise_belief, dim1=-2, dim2=-1).sum(dim=-1, keepdim=True)
        conflict = total_mass - agreement_mass
        normalizer = (1.0 - conflict).clamp(min=self.eps)

        fused_belief = (
            belief1 * belief2
            + belief1 * uncertainty2
            + uncertainty1 * belief2
        ) / normalizer
        fused_uncertainty = (uncertainty1 * uncertainty2) / normalizer

        fused_strength = self.num_classes / fused_uncertainty.clamp(min=self.eps)
        fused_evidence = fused_belief * fused_strength
        fused_alpha = fused_evidence + 1.0
        return fused_alpha, fused_belief, fused_uncertainty, conflict


class DecoupledBCAEDL(nn.Module):
    def __init__(
        self,
        text_model_name=HP.TEXT_MODEL_NAME,
        num_classes=HP.NUM_CLASSES,
        proj_dim=HP.PROJ_DIM,
        attn_heads=HP.ATTN_HEADS,
        dropout=HP.DROPOUT,
        use_pretrained_image=HP.USE_PRETRAINED_IMAGE,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.proj_dim = proj_dim

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_hidden = self.text_encoder.config.hidden_size
        self.text_projection = nn.Sequential(
            nn.Linear(text_hidden, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        image_weights = models.ResNet50_Weights.IMAGENET1K_V2 if use_pretrained_image else None
        try:
            resnet = models.resnet50(weights=image_weights)
        except Exception as exc:
            print(f"Warning: failed to load pretrained ResNet50 weights ({exc}). Falling back to random weights.")
            resnet = models.resnet50(weights=None)
        self.image_backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.image_projection = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        self.text_decoupler = FeatureDecoupler(proj_dim, dropout)
        self.image_decoupler = FeatureDecoupler(proj_dim, dropout)

        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=proj_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=proj_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.t2v_norm = nn.LayerNorm(proj_dim)
        self.v2t_norm = nn.LayerNorm(proj_dim)
        self.t2v_edl = DirectionalEDLHead(proj_dim, num_classes, dropout)
        self.v2t_edl = DirectionalEDLHead(proj_dim, num_classes, dropout)
        self.fusion = DempsterShaferFusion(num_classes)

    def extract_text_tokens(self, input_ids, attention_mask):
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.text_projection(outputs.last_hidden_state)

    def extract_image_tokens(self, images):
        feature_map = self.image_backbone(images)
        batch_size, channels, height, width = feature_map.shape
        tokens = feature_map.view(batch_size, channels, height * width).transpose(1, 2)
        return self.image_projection(tokens)

    def forward(self, input_ids, attention_mask, images):
        text_tokens = self.extract_text_tokens(input_ids, attention_mask)
        image_tokens = self.extract_image_tokens(images)

        text_sent, text_background = self.text_decoupler(text_tokens)
        image_sent, image_background = self.image_decoupler(image_tokens)

        key_padding_mask = attention_mask == 0

        # Text-to-image: text queries attend visual sentiment tokens.
        t2v_tokens, t2v_weights = self.text_to_image_attn(
            query=text_sent,
            key=image_sent,
            value=image_sent,
            need_weights=True,
            average_attn_weights=False,
        )
        t2v_tokens = self.t2v_norm(t2v_tokens + text_sent)
        t2v_feature_raw = masked_mean_pool(t2v_tokens, attention_mask)

        # Image-to-text: visual queries attend textual sentiment tokens.
        v2t_tokens, v2t_weights = self.image_to_text_attn(
            query=image_sent,
            key=text_sent,
            value=text_sent,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        v2t_tokens = self.v2t_norm(v2t_tokens + image_sent)
        v2t_feature_raw = v2t_tokens.mean(dim=1)

        # EDL uncertainty gates. Raw uncertainty controls how much each directional feature contributes.
        _, _, u_t2v_raw, _ = self.t2v_edl(t2v_feature_raw)
        _, _, u_v2t_raw, _ = self.v2t_edl(v2t_feature_raw)
        t2v_feature = t2v_feature_raw * (1.0 - u_t2v_raw)
        v2t_feature = v2t_feature_raw * (1.0 - u_v2t_raw)

        alpha_t2v, belief_t2v, u_t2v, evidence_t2v = self.t2v_edl(t2v_feature)
        alpha_v2t, belief_v2t, u_v2t, evidence_v2t = self.v2t_edl(v2t_feature)
        alpha_fused, belief_fused, u_fused, conflict = self.fusion(alpha_t2v, alpha_v2t)

        text_ortho = orthogonal_penalty(text_sent, text_background, attention_mask)
        image_ortho = orthogonal_penalty(image_sent, image_background)
        orthogonal_loss = 0.5 * (text_ortho + image_ortho)

        preds = belief_fused.argmax(dim=-1)

        return {
            "alpha_t2v": alpha_t2v,
            "alpha_v2t": alpha_v2t,
            "belief_t2v": belief_t2v,
            "belief_v2t": belief_v2t,
            "u_t2v": u_t2v,
            "u_v2t": u_v2t,
            "alpha_fused": alpha_fused,
            "belief_fused": belief_fused,
            "u_fused": u_fused,
            "preds": preds,
            "orthogonal_loss": orthogonal_loss,
            "conflict": conflict,
            "evidence_t2v": evidence_t2v,
            "evidence_v2t": evidence_v2t,
            "t2v_attention": t2v_weights,
            "v2t_attention": v2t_weights,
        }


model = DecoupledBCAEDL().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## Fase 7 - EDL Loss, Optimizer, dan Regularisasi

Loss utama memakai EDL expected MSE + KL annealing. Dua directional head diberi auxiliary loss, sedangkan orthogonal loss menjaga fitur sentimen/background tidak runtuh ke representasi yang sama.


In [ ]:
# ============================================================
# CELL 6: LOSS FUNCTIONS AND OPTIMIZER
# ============================================================

def one_hot(labels, num_classes=HP.NUM_CLASSES):
    return F.one_hot(labels, num_classes=num_classes).float()


def edl_expected_mse_loss(alpha, target_onehot, class_weights=None, eps=1e-8):
    strength = alpha.sum(dim=-1, keepdim=True)
    probs = alpha / strength.clamp(min=eps)
    squared_error = (target_onehot - probs).pow(2)
    variance = probs * (1.0 - probs) / (strength + 1.0).clamp(min=eps)
    sample_loss = (squared_error + variance).sum(dim=-1)

    if class_weights is not None:
        sample_weights = (target_onehot * class_weights.unsqueeze(0)).sum(dim=-1)
        sample_loss = sample_loss * sample_weights

    return sample_loss.mean()


def edl_kl_divergence(alpha, target_onehot, eps=1e-8):
    alpha_tilde = target_onehot + (1.0 - target_onehot) * alpha
    beta = torch.ones_like(alpha_tilde)

    sum_alpha = alpha_tilde.sum(dim=-1, keepdim=True)
    sum_beta = beta.sum(dim=-1, keepdim=True)

    log_norm_alpha = torch.lgamma(sum_alpha) - torch.lgamma(alpha_tilde.clamp(min=eps)).sum(dim=-1, keepdim=True)
    log_norm_beta = torch.lgamma(beta).sum(dim=-1, keepdim=True) - torch.lgamma(sum_beta)
    digamma_term = ((alpha_tilde - beta) * (torch.digamma(alpha_tilde.clamp(min=eps)) - torch.digamma(sum_alpha))).sum(dim=-1, keepdim=True)
    kl = log_norm_alpha + log_norm_beta + digamma_term
    return kl.mean()


def annealing_coefficient(epoch, annealing_epochs=HP.KL_ANNEALING_EPOCHS):
    return min(1.0, float(epoch + 1) / float(max(1, annealing_epochs)))


def total_edl_loss(outputs, labels, epoch, class_weights=None):
    targets = one_hot(labels, HP.NUM_CLASSES)
    anneal = annealing_coefficient(epoch, HP.KL_ANNEALING_EPOCHS)

    fused_cls = edl_expected_mse_loss(outputs["alpha_fused"], targets, class_weights)
    t2v_cls = edl_expected_mse_loss(outputs["alpha_t2v"], targets, class_weights)
    v2t_cls = edl_expected_mse_loss(outputs["alpha_v2t"], targets, class_weights)

    fused_kl = edl_kl_divergence(outputs["alpha_fused"], targets)
    t2v_kl = edl_kl_divergence(outputs["alpha_t2v"], targets)
    v2t_kl = edl_kl_divergence(outputs["alpha_v2t"], targets)

    auxiliary_loss = 0.5 * (t2v_cls + v2t_cls)
    kl_loss = fused_kl + HP.AUX_LOSS_WEIGHT * 0.5 * (t2v_kl + v2t_kl)
    ortho_loss = outputs["orthogonal_loss"]

    loss = (
        fused_cls
        + HP.AUX_LOSS_WEIGHT * auxiliary_loss
        + HP.KL_WEIGHT * anneal * kl_loss
        + HP.ORTHO_LOSS_WEIGHT * ortho_loss
    )

    return loss, {
        "loss": loss.detach().item(),
        "fused_cls": fused_cls.detach().item(),
        "aux_cls": auxiliary_loss.detach().item(),
        "kl": kl_loss.detach().item(),
        "anneal": anneal,
        "orthogonal": ortho_loss.detach().item(),
    }


def set_backbone_trainable(model, trainable=True):
    for param in model.text_encoder.parameters():
        param.requires_grad = trainable
    for param in model.image_backbone.parameters():
        param.requires_grad = trainable


def build_optimizer(model):
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("text_encoder") or name.startswith("image_backbone"):
            backbone_params.append(param)
        else:
            head_params.append(param)

    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": HP.BACKBONE_LR},
            {"params": head_params, "lr": HP.HEAD_LR},
        ],
        weight_decay=HP.WEIGHT_DECAY,
    )


optimizer = build_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, HP.MAX_EPOCHS))
print("Loss functions, optimizer, and scheduler are ready.")


## Fase 8 - Training, Evaluasi, dan Analisis Uncertainty


In [ ]:
# ============================================================
# CELL 7: TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def move_batch_to_device(batch, device):
    return {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
        "image": batch["image"].to(device),
        "label": batch["label"].to(device),
    }


def summarize_uncertainty(labels, preds, uncertainties):
    labels = np.asarray(labels)
    preds = np.asarray(preds)
    uncertainties = np.asarray(uncertainties)
    correct_mask = labels == preds
    wrong_mask = ~correct_mask
    return {
        "uncertainty_correct": float(uncertainties[correct_mask].mean()) if correct_mask.any() else float("nan"),
        "uncertainty_wrong": float(uncertainties[wrong_mask].mean()) if wrong_mask.any() else float("nan"),
    }


def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    set_backbone_trainable(model, trainable=epoch >= HP.BACKBONE_FREEZE_EPOCHS)

    running_loss = 0.0
    all_labels, all_preds, all_uncertainties = [], [], []
    pbar = tqdm(loader, desc=f"Epoch {epoch + 1} [Train]", leave=False)

    for batch in pbar:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch["input_ids"], batch["attention_mask"], batch["image"])
        loss, loss_parts = total_edl_loss(outputs, batch["label"], epoch, class_weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), HP.GRAD_CLIP_NORM)
        optimizer.step()

        batch_size = batch["label"].size(0)
        running_loss += loss.item() * batch_size
        all_labels.extend(batch["label"].detach().cpu().numpy().tolist())
        all_preds.extend(outputs["preds"].detach().cpu().numpy().tolist())
        all_uncertainties.extend(outputs["u_fused"].detach().squeeze(-1).cpu().numpy().tolist())
        pbar.set_postfix(loss=f"{loss.item():.4f}", anneal=f"{loss_parts['anneal']:.2f}")

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }
    metrics.update(summarize_uncertainty(all_labels, all_preds, all_uncertainties))
    return metrics


@torch.no_grad()
def evaluate(model, loader, epoch=0, return_details=False):
    model.eval()
    running_loss = 0.0
    all_labels, all_preds, all_uncertainties, all_conflicts = [], [], [], []

    for batch in tqdm(loader, desc="Evaluate", leave=False):
        batch = move_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"], batch["image"])
        loss, _ = total_edl_loss(outputs, batch["label"], epoch, class_weights)

        batch_size = batch["label"].size(0)
        running_loss += loss.item() * batch_size
        all_labels.extend(batch["label"].detach().cpu().numpy().tolist())
        all_preds.extend(outputs["preds"].detach().cpu().numpy().tolist())
        all_uncertainties.extend(outputs["u_fused"].detach().squeeze(-1).cpu().numpy().tolist())
        all_conflicts.extend(outputs["conflict"].detach().squeeze(-1).cpu().numpy().tolist())

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
        "mean_conflict": float(np.mean(all_conflicts)),
    }
    metrics.update(summarize_uncertainty(all_labels, all_preds, all_uncertainties))

    if return_details:
        metrics.update({
            "labels": np.asarray(all_labels),
            "preds": np.asarray(all_preds),
            "uncertainties": np.asarray(all_uncertainties),
            "conflicts": np.asarray(all_conflicts),
        })

    return metrics


class EarlyStopping:
    def __init__(self, patience=HP.EARLY_STOP_PATIENCE, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = -float("inf")
        self.best_state = None
        self.counter = 0
        self.should_stop = False

    def step(self, score, model):
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            self.should_stop = self.counter >= self.patience
        return self.should_stop


print("Training and evaluation helpers are ready.")


In [ ]:
# ============================================================
# CELL 8: TRAINING LOOP
# ============================================================

history = {
    "train_loss": [],
    "train_accuracy": [],
    "train_macro_f1": [],
    "train_weighted_f1": [],
    "train_uncertainty_correct": [],
    "train_uncertainty_wrong": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "val_weighted_f1": [],
    "val_uncertainty_correct": [],
    "val_uncertainty_wrong": [],
    "val_mean_conflict": [],
}

early_stopping = EarlyStopping()

print("=" * 72)
print("TRAINING START - Decoupled BCA + EDL + Dempster-Shafer Fusion")
print("=" * 72)

for epoch in range(HP.MAX_EPOCHS):
    train_metrics = train_one_epoch(model, train_loader, optimizer, epoch)
    val_metrics = evaluate(model, val_loader, epoch=epoch)
    scheduler.step()

    for key in ["loss", "accuracy", "macro_f1", "weighted_f1", "uncertainty_correct", "uncertainty_wrong"]:
        history[f"train_{key}"].append(train_metrics[key])
        history[f"val_{key}"].append(val_metrics[key])
    history["val_mean_conflict"].append(val_metrics["mean_conflict"])

    print(
        f"Epoch {epoch + 1:02d}/{HP.MAX_EPOCHS} | "
        f"Train L={train_metrics['loss']:.4f} F1={train_metrics['macro_f1']:.4f} Acc={train_metrics['accuracy']:.4f} | "
        f"Val L={val_metrics['loss']:.4f} F1={val_metrics['macro_f1']:.4f} Acc={val_metrics['accuracy']:.4f} | "
        f"u_wrong={val_metrics['uncertainty_wrong']:.4f} conflict={val_metrics['mean_conflict']:.4f}"
    )

    if early_stopping.step(val_metrics["macro_f1"], model):
        print(f"Early stopping at epoch {epoch + 1}. Best Val Macro-F1: {early_stopping.best_score:.4f}")
        break

if early_stopping.best_state is not None:
    model.load_state_dict(early_stopping.best_state)
    print(f"Loaded best model state with Val Macro-F1: {early_stopping.best_score:.4f}")

print("=" * 72)
print("TRAINING COMPLETE")
print("=" * 72)


In [ ]:
# ============================================================
# CELL 9: TRAINING CURVES
# ============================================================

if history["train_loss"]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_macro_f1"], label="Train Macro-F1")
    axes[1].plot(history["val_macro_f1"], label="Val Macro-F1")
    axes[1].set_title("Macro-F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(history["val_uncertainty_correct"], label="Val Uncertainty Correct")
    axes[2].plot(history["val_uncertainty_wrong"], label="Val Uncertainty Wrong")
    axes[2].set_title("Validation Uncertainty")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Training history is empty. Run the training cell first.")


In [ ]:
# ============================================================
# CELL 10: TEST SET EVALUATION
# ============================================================

test_metrics = evaluate(model, test_loader, epoch=HP.MAX_EPOCHS - 1, return_details=True)

print("=" * 72)
print("TEST SET RESULTS")
print("=" * 72)
print(f"Accuracy:              {test_metrics['accuracy']:.4f}")
print(f"Macro-F1:              {test_metrics['macro_f1']:.4f}")
print(f"Weighted-F1:           {test_metrics['weighted_f1']:.4f}")
print(f"Mean conflict:         {test_metrics['mean_conflict']:.4f}")
print(f"Uncertainty - correct: {test_metrics['uncertainty_correct']:.4f}")
print(f"Uncertainty - wrong:   {test_metrics['uncertainty_wrong']:.4f}")
print()

target_names = [id2label[i] for i in range(HP.NUM_CLASSES)] if "id2label" in globals() else ["negative", "neutral", "positive"]
print(classification_report(test_metrics["labels"], test_metrics["preds"], target_names=target_names, digits=4))

cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix - Decoupled BCA + EDL")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
correctness = np.where(test_metrics["labels"] == test_metrics["preds"], "Correct", "Wrong")
sns.boxplot(x=correctness, y=test_metrics["uncertainties"], ax=ax, palette="Set2")
ax.set_title("Fused Uncertainty by Prediction Correctness")
ax.set_xlabel("Prediction")
ax.set_ylabel("Fused uncertainty")
plt.tight_layout()
plt.show()


## Optional Checks dan Save

Cell berikut aman untuk `Run All` karena default flag-nya `False`. Aktifkan saat ingin melakukan smoke test atau menyimpan model terbaik.


In [ ]:
# ============================================================
# CELL 11: OPTIONAL SMOKE CHECKS (DISABLED BY DEFAULT)
# ============================================================

RUN_SMOKE_CHECKS = False

if RUN_SMOKE_CHECKS:
    sample = train_dataset[0]
    print("Single sample shapes:")
    print("input_ids:", tuple(sample["input_ids"].shape))
    print("attention_mask:", tuple(sample["attention_mask"].shape))
    print("image:", tuple(sample["image"].shape))
    print("label:", sample["label"].item())

    smoke_batch_raw = next(iter(train_loader))
    smoke_batch = move_batch_to_device(smoke_batch_raw, device)
    model.eval()
    smoke_outputs = model(smoke_batch["input_ids"], smoke_batch["attention_mask"], smoke_batch["image"])
    smoke_loss, smoke_loss_parts = total_edl_loss(smoke_outputs, smoke_batch["label"], epoch=0, class_weights=class_weights)

    print("\nForward-pass checks:")
    print("alpha_t2v:", tuple(smoke_outputs["alpha_t2v"].shape))
    print("alpha_v2t:", tuple(smoke_outputs["alpha_v2t"].shape))
    print("alpha_fused:", tuple(smoke_outputs["alpha_fused"].shape))
    print("u_fused:", tuple(smoke_outputs["u_fused"].shape))
    print("preds:", tuple(smoke_outputs["preds"].shape))
    print("finite loss:", torch.isfinite(smoke_loss).item(), smoke_loss.item())
    print("loss parts:", smoke_loss_parts)
else:
    print("Smoke checks are disabled. Set RUN_SMOKE_CHECKS = True to run a one-batch validation.")


In [ ]:
# ============================================================
# CELL 12: OPTIONAL SAVE ARTIFACTS (DISABLED BY DEFAULT)
# ============================================================

SAVE_ARTIFACTS = False

if SAVE_ARTIFACTS:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "hp": {key: value for key, value in HP.__dict__.items() if key.isupper()},
        "label_map": label_map if "label_map" in globals() else {"negative": 0, "neutral": 1, "positive": 2},
        "id2label": id2label if "id2label" in globals() else {0: "negative", 1: "neutral", 2: "positive"},
        "history": history,
    }
    torch.save(checkpoint, HP.BEST_MODEL_PATH)
    np.save(HP.HISTORY_PATH, history, allow_pickle=True)
    print(f"Saved checkpoint to {HP.BEST_MODEL_PATH}")
    print(f"Saved history to {HP.HISTORY_PATH}")
else:
    print("Artifact saving is disabled. Set SAVE_ARTIFACTS = True after training if you want to persist outputs.")
